In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import mne
from specparam import SpectralGroupModel

mne.set_log_level("ERROR")

# 1. SETUP
BIDS_ROOT  = "/Users/elizabethkaplan/Desktop/ds007615"
DERIV_ROOT = os.path.join(BIDS_ROOT, "derivatives", "preproc")   # where clean-epo.fif live
OUT_DIR    = os.path.join(DERIV_ROOT, "specparam")
REPORT_DIR = os.path.join(OUT_DIR, "reports")
TASK       = "rest"
ACQS       = ["ec", "eo"]

# PSD
PSD_FMIN, PSD_FMAX = 1.0, 45.0

# specparam
FIT_RANGE       = (1.0, 40.0)
APERIODIC_MODE  = "fixed"        # "fixed" vs "knee"
PEAK_WIDTH      = (1.0, 8.0)
MAX_N_PEAKS     = 6
MIN_PEAK_HEIGHT = 0.10

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# FUNCTIONZ
def _metric(group_result, keys):
    """Robustly pull a metric (r^2 / error) out of a specparam FitResults object."""
    m = getattr(group_result, "metrics", {}) or {}
    for k in keys:
        if k in m:
            return m[k]
    return np.nan


def analyze_run(epo_path, sub, acq):
    """specparam on one subject/condition. Returns (per-channel DataFrame, report string)."""
    epochs = mne.read_epochs(epo_path, preload=True, verbose=False)
    epochs.pick("eeg")

    n_seg = len(epochs.times)                     # full epoch as one Welch segment
    spectrum = epochs.compute_psd(method="welch", fmin=PSD_FMIN, fmax=PSD_FMAX,
                                  n_fft=n_seg, n_per_seg=n_seg, verbose=False)
    psds, freqs = spectrum.get_data(return_freqs=True)   # (n_epochs, n_channels, n_freqs)
    n_ep, n_ch, n_f = psds.shape

    # fit every epoch × channel spectrum at once, then reshape back
    fg = SpectralGroupModel(peak_width_limits=PEAK_WIDTH, max_n_peaks=MAX_N_PEAKS,
                            min_peak_height=MIN_PEAK_HEIGHT,
                            aperiodic_mode=APERIODIC_MODE, verbose=False)
    fg.fit(freqs, psds.reshape(n_ep * n_ch, n_f), freq_range=FIT_RANGE)

    exp = fg.get_params("aperiodic", "exponent").reshape(n_ep, n_ch)
    off = fg.get_params("aperiodic", "offset").reshape(n_ep, n_ch)
    r2  = np.array([_metric(g, ["gof_rsquared", "r_squared"])
                    for g in fg.results.group_results]).reshape(n_ep, n_ch)
    err = np.array([_metric(g, ["error_mae", "error"])
                    for g in fg.results.group_results]).reshape(n_ep, n_ch)
    knee = (fg.get_params("aperiodic", "knee").reshape(n_ep, n_ch)
            if APERIODIC_MODE == "knee" else None)

    # average across epochs -> one row per channel
    rows = []
    for ci, ch in enumerate(spectrum.ch_names):
        row = dict(subject=sub, acq=acq, channel=ch,
                   exponent=np.nanmean(exp[:, ci]),
                   exponent_sd=np.nanstd(exp[:, ci]),
                   offset=np.nanmean(off[:, ci]),
                   offset_sd=np.nanstd(off[:, ci]),
                   r_squared=np.nanmean(r2[:, ci]),
                   error_mae=np.nanmean(err[:, ci]),
                   n_epochs=n_ep)
        if knee is not None:
            row["knee"] = np.nanmean(knee[:, ci])
        rows.append(row)
    df = pd.DataFrame(rows)

    # text report of fit metrics
    lines = [
        f"specparam fit report — sub-{sub}  task-{TASK}  acq-{acq}",
        "=" * 60,
        f"epochs fit           : {n_ep}",
        f"channels             : {n_ch}",
        f"frequency resolution : {freqs[1] - freqs[0]:.3f} Hz",
        f"fit range            : {FIT_RANGE[0]}-{FIT_RANGE[1]} Hz",
        f"aperiodic mode       : {APERIODIC_MODE}",
        f"mean R^2             : {np.nanmean(r2):.4f}   (min {np.nanmin(r2):.3f})",
        f"mean MAE             : {np.nanmean(err):.4f}",
        f"mean exponent        : {df['exponent'].mean():.4f}",
        f"mean offset          : {df['offset'].mean():.4f}",
        "",
        f"{'channel':<10}{'exponent':>10}{'offset':>10}{'R^2':>8}{'MAE':>8}",
        "-" * 46,
    ]
    for _, r in df.iterrows():
        lines.append(f"{r['channel']:<10}{r['exponent']:>10.3f}{r['offset']:>10.3f}"
                     f"{r['r_squared']:>8.3f}{r['error_mae']:>8.3f}")
    return df, "\n".join(lines)


# RUN 
frames, run_summary = [], []
for acq in ACQS:
    pattern = os.path.join(DERIV_ROOT, "sub-*",
                           f"sub-*_task-{TASK}_acq-{acq}_clean-epo.fif")
    for epo_path in sorted(glob.glob(pattern)):
        fname = os.path.basename(epo_path)
        sub = fname.split("_")[0].replace("sub-", "")
        try:
            df, report = analyze_run(epo_path, sub, acq)
            frames.append(df)
            with open(os.path.join(REPORT_DIR,
                      f"sub-{sub}_task-{TASK}_acq-{acq}_specparam.txt"), "w") as f:
                f.write(report)
            line = (f"sub-{sub} {acq}: {df['n_epochs'].iloc[0]} epochs | "
                    f"mean exp {df['exponent'].mean():.2f} | "
                    f"mean R^2 {df['r_squared'].mean():.3f}")
        except Exception as e:
            line = f"sub-{sub} {acq}: ERROR {type(e).__name__}: {e}"
        print(line)
        run_summary.append(line)

# master table: aperiodic exponent + offset per subject × channel × condition
df_all = pd.concat(frames, ignore_index=True)
csv_path = os.path.join(OUT_DIR, "aperiodic_per_subject_channel.csv")
df_all.to_csv(csv_path, index=False)

with open(os.path.join(OUT_DIR, "specparam_run_summary.txt"), "w") as f:
    f.write("\n".join(run_summary))

print(f"\nSaved {csv_path}  {df_all.shape}")
print(f"Reports -> {REPORT_DIR}")

sub-01 ec: 98 epochs | mean exp 1.40 | mean R^2 0.623
